# 2.4 - Views in Databricks

This notebook demonstrates how to work with views, materialized views,
and temporary views in Databricks SQL. It includes examples
of creating, querying, and managing views to enable flexible data analysis. 

**Topics covered:**
- Creating permanent and temporary views
- Filtering and selecting data with views
- Materialized views
- Global temporary views
- Example queries and troubleshooting

---

In [0]:
%sql
-- Set default catalog for subsequent queries
USE CATALOG databricks_demo;

In [0]:
%sql
-- Create a sample table with smartphone data
-- Insert sample smartphone records for view demonstrations
CREATE TABLE IF NOT EXISTS tb_smart_phones
(
  id INT,
  name STRING,
  brand STRING,
  price FLOAT,
  rating FLOAT,
  rating_count INT,
  year INT
);

-- Insert sample smartphone data
INSERT INTO tb_smart_phones
VALUES
(1, 'iPhone 12', 'Apple', 799, 4.6, 1000000, 2020),
-- ...
(43, 'OnePlus 11 Pro', 'OnePlus', 969, 4.8, 350000, 2023); -- truncated for brevity

In [0]:
# Display sample smartphone data to verify table creation
spark.sql(
  """
  select * from tb_smart_phones
  """).show(2)

In [0]:
%sql
-- Show existing tables in the current catalog/schema
SHOW TABLES

In [0]:
%sql
-- Show available views in the current schema
SHOW VIEWS

In [0]:
%sql
-- List materialized views in the catalog
SELECT * FROM databricks_demo.information_schema.views
WHERE is_materialized = 'YES';

In [0]:
# Create a permanent view of Apple smartphones
spark.sql(
    """
    CREATE OR REPLACE VIEW vw_apple_smart_phones
    AS
    SELECT *
    FROM tb_smart_phones
    WHERE lower(brand) = 'apple'
    """
)

In [0]:
%sql
-- Query the permanent Apple smartphone view
SELECT *
FROM vw_apple_smart_phones;

In [0]:
%sql
-- Show tables to verify existence of source and view
SHOW TABLES;

In [0]:
%sql
-- Show all views in the schema
SHOW VIEWS;

In [0]:
%sql
-- Create a temporary view of distinct smartphone brands
CREATE OR REPLACE TEMP VIEW vw_tmp_smart_phone_brands
AS
SELECT DISTINCT
  brand
FROM tb_smart_phones;

In [0]:
%sql
-- Show views, including temporary ones
SHOW VIEWS

In [0]:
%sql
-- Query the temporary brands view
SELECT * FROM vw_tmp_smart_phone_brands;

In [0]:
%sql
-- Attempt to create a global temporary view for latest smartphone brands (post-2023)
-- Note: Not supported with serverless compute
CREATE OR REPLACE GLOBAL TEMP VIEW vw_tmp_smart_phone_brands_latest
AS
SELECT DISTINCT
  brand
FROM tb_smart_phones
WHERE year > 2023;

In [0]:
%sql
-- Show tables in global_temp to check global temp views
SHOW TABLES IN global_temp;

In [0]:
%sql
-- Query the global temporary view for latest brands
SELECT * FROM global_temp.vw_tmp_smart_phone_brands_latest;

In [0]:
# Try to create a materialized view; fallback to regular view if not supported
try:
    spark.sql("""
        CREATE MATERIALIZED VIEW IF NOT EXISTS apple_smart_phones_mv AS
        SELECT * FROM tb_smart_phones WHERE lower(brand) = 'apple'
    """)
except Exception as e:
    print(e)
    spark.sql("""
        CREATE OR REPLACE VIEW apple_smart_phones_mv AS
        SELECT * FROM tb_smart_phones WHERE lower(brand) = 'apple'
    """)

In [0]:
%sql
SHOW VIEWS